In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax
from flax import nnx
import orbax.checkpoint as ocp

from jaxpm import camels, data, plotting, graph
from jaxpm.painting import cic_paint, cic_read
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN

import optuna

jax.devices("gpu")

[cuda(id=0)]

# functions

## training

In [3]:
# @nnx.jit
# def train_step(model, optimizer, x, y):

#     def loss_fn(model):
#         y_pred = model(*x)
#         return jnp.mean((y - y_pred) **2)

#     loss, grads = nnx.value_and_grad(loss_fn)(model)
#     optimizer.update(grads)

#     return loss

# @nnx.jit
# def vali_loop(model, X, Y):
#     losses = []
#     for i in range(scales.shape[0]):
#         x, y = tuple(x[i] for x in X), Y[i]
        
#         losses.append(jnp.mean((y - model(*x))**2))
#     losses = jnp.mean(jnp.stack(losses))

#     return losses
    
# # NOTE this is GD, not SGD. There's no batches, all particles in the snapshot are evaluated every step
# def train_model(
#     model, 
#     X,
#     Y,
#     X_vali,
#     Y_vali,
#     total_steps=3_000,
#     vali_every=100,
#     learning_rate=1e-3,
#     cosine_decay=True,
# ):
#     optimizer = optax.chain(
#         optax.clip_by_global_norm(1),
#         optax.adam(learning_rate)
#     )
#     optimizer = nnx.Optimizer(model, optimizer)

#     if cosine_decay:
#         learning_rate = optax.cosine_decay_schedule(
#             init_value=learning_rate, 
#             decay_steps=total_steps, 
#             alpha=0.1
#         )

#     losses = []
#     vali_steps = []
#     vali_losses = []
#     vali_loss = np.inf
#     for i in (pbar := tqdm.tqdm(range(total_steps))):
#         # select single snapshot
#         j = np.random.choice(np.arange(scales.shape[0]))
#         x, y = tuple(x[j] for x in X), Y[j]
        
#         loss = train_step(model, optimizer, x, y)
#         losses.append(loss)

#         if (i % vali_every == 0) and (i != 0) or i == total_steps - 1:
#             vali_steps.append(i)
#             vali_loss = vali_loop(model, X_vali, Y_vali)
#             vali_losses.append(vali_loss)

#         pbar.set_description(f"train={loss:.4f}, vali={vali_loss:.4f}")

#     fig, ax = plt.subplots()
#     ax.plot(losses, label="training")
#     ax.plot(vali_steps, vali_losses, label="validation")
#     ax.legend(loc="upper right")
#     ax.set(yscale="log")
    

## plotting

In [4]:
def plot_preds(pred, label, pred_vali, label_vali):
    nrows = pred_vali.shape[0]
    ncols = 2
    
    fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=(2*ncols,2*nrows))

    # fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=(2*ncols,2*nrows))
    # ncols = y_true.shape[-1]

    for i in range(nrows):
        ax[i,0].scatter(label[i], pred[i], s=0.1, color="tab:blue")
    
        ax[i,0].plot([label[i].min(), label[i].max()], [label[i].min(), label[i].max()], color="k", linestyle="--")
        ax[i,0].set_aspect("equal")
        ax[i,0].set_box_aspect(1)
        # ax[i].set_yticks(ax[i].get_xticks())

    ax[0,0].set(title="training")
    
    for i in range(nrows):
        ax[i,1].scatter(label_vali[i], pred_vali[i], s=0.1, color="tab:orange")
    
        ax[i,1].plot([label_vali[i].min(), label_vali[i].max()], [label_vali[i].min(), label_vali[i].max()], color="k", linestyle="--")
        ax[i,1].set_aspect("equal")
        ax[i,1].set_box_aspect(1)
        # ax[i].set_yticks(ax[i].get_xticks())
    
    ax[0,1].set(title="validation")

    fig.tight_layout()

In [5]:
def normalized_paint(positions, weights):
    N_pos = cic_paint(jnp.zeros([mesh_per_dim] * 3), positions)
    pos_N = cic_read(N_pos, positions)

    return cic_paint(jnp.zeros(mesh_shape), positions, weights/pos_N)


def plot_field(field, ax, vlims=None):
    log_sum = jnp.log10(field.sum(axis=0))

    if vlims is None:
        vmin, vmax = log_sum.min(), log_sum.max()
        vlims = (vmin, vmax)
    else:
        vmin, vmax = vlims
    
    return ax.imshow(log_sum, cmap="magma", vmin=vmin, vmax=vmax), vlims, log_sum


def plot_comparison(true_field, pred_field, true_label="CAMELS", pred_label="", quantity_label="P", suptitle=""):
    fig, ax = plt.subplots(figsize=(3*6, 6+1), ncols=3)
    
    im, vlims, true_field_log_sum = plot_field(true_field, ax[0])
    ax[0].set(title=true_label)
    
    im, _, pred_field_log_sum = plot_field(pred_field, ax[1], vlims=vlims)
    ax[1].set(title=pred_label)
    
    fig.colorbar(im, ax=ax[:2], label=f"log(sum({quantity_label}))", orientation="horizontal", shrink=0.6, aspect=20)
    
    residual = true_field_log_sum - pred_field_log_sum
    lim = np.max(np.abs(residual[np.isfinite(residual)]))
    print(lim)
    im = ax[2].imshow(residual, cmap="seismic", vmin=-lim, vmax=lim)
    ax[2].set(title="residual (true - pred)")
    fig.colorbar(im, ax=ax[2], label=f"log(sum({quantity_label}_1)) - log(sum({quantity_label}_2))", orientation="horizontal", shrink=0.6, aspect=10)
    
    for i in range(len(ax)):
        ax[i].set_yticks([])
        ax[i].set_xticks([])
    
    fig.suptitle(suptitle, fontsize=16)

In [6]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3

## CAMELS snapshots

In [7]:
# i_snapshots = [-2, -1]
i_snapshots = range(1, 33+4, 4)
# i_snapshots = None

In [8]:
SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0"

train_dict = camels.load_CV_snapshots(
    SIM,
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
)

Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_034.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_050.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_066.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_082.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


loading snapshots: 100%|██████████| 9/9 [01:45<00:00, 11.78s/it]


In [9]:
SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1"

test_dict = camels.load_CV_snapshots(
    SIM,
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
)

Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_034.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_050.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_066.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_082.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


loading snapshots: 100%|██████████| 9/9 [01:43<00:00, 11.52s/it]


In [10]:
n_scales = len(train_dict["scales"])

@nnx.jit
def train_step(model, optimizer, x, y):

    def loss_fn(model):
        y_pred = model(*x)
        return jnp.mean((y - y_pred) **2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

@nnx.jit
def vali_loop(model, X, Y):
    losses = []
    for i in range(n_scales):
        x, y = tuple(x[i] for x in X), Y[i]

        losses.append(jnp.mean((y - model(*x))**2))

    losses = jnp.mean(jnp.stack(losses))

    return losses

def train_loop(
    model, 
    X, 
    Y,
    X_vali,
    Y_vali,
    n_steps,
    learning_rate,
    clip_norm,
    cosine_sched,
):
    if cosine_sched:
        learning_rate = optax.cosine_decay_schedule(
            init_value=learning_rate, 
            decay_steps=n_steps, 
            alpha=0.1
        )

    if clip_norm:
        optimizer = optax.chain(
            optax.clip_by_global_norm(1),
            optax.adam(learning_rate)
        )
    else:
        optimizer = optax.adam(learning_rate)
        
    optimizer = nnx.Optimizer(model, optimizer)

    for i in (pbar := tqdm.tqdm(range(n_steps))):
        # select single snapshot
        j = np.random.choice(np.arange(n_scales))
        x, y = tuple(x[j] for x in X), Y[j]
        
        loss = train_step(model, optimizer, x, y)
        pbar.set_description(f"train={loss:.4f}")

    vali_loss = vali_loop(model, X_vali, Y_vali)

    return vali_loss


# MLP

In [11]:
X, _, Y, _ = data.get_offline_regression_data(train_dict, y_labels=["P"])
X_vali, _, Y_vali, _ = data.get_offline_regression_data(test_dict, y_labels=["P"])

def mlp_training(
    # net
    d_hidden,
    n_hidden,
    dropout_rate,
    activation,
    # optimizer
    n_steps,
    learning_rate,
    clip_norm,
    cosine_sched,
):
    if activation == "relu":
        activation = jax.nn.relu
    elif activation == "leaky_relu":
        activation = jax.nn.leaky_relu
    elif activation == "sigmoid":
        activation = jax.nn.sigmoid
    
    model = MLP(
        X.shape[-1], 
        Y.shape[-1], 
        d_hidden, 
        n_hidden,
        nnx.Rngs(0),
        dropout_rate,
        activation,
    )

    vali_loss = train_loop(
        model, 
        (X,), 
        Y,
        (X_vali,),
        Y_vali,
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )

    return vali_loss

def mlp_objective(trial):
    d_hidden = trial.suggest_categorical("d_hidden", [16, 32, 64, 128])
    n_hidden = trial.suggest_int("n_hidden", 1, 8)
    dropout_rate = trial.suggest_float("dropout_rate", 1e-3, 1e-1, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "sigmoid", "leaky_relu"])
                           
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    clip_norm = trial.suggest_categorical("clip_norm", [True, False])
    cosine_sched = trial.suggest_categorical("cosine_sched", [True, False])

    n_steps = 3_000
    # n_steps = trial.suggest_int("train_steps", 1000, 100000, log=True)

    val_loss = mlp_training(
        d_hidden,
        n_hidden,
        dropout_rate,
        activation,
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )
    return val_loss

In [12]:
study = optuna.create_study(direction="minimize")
study.optimize(mlp_objective, n_trials=20)

[I 2025-02-19 16:27:49,682] A new study created in memory with name: no-name-9c90d157-2de3-493f-9a2d-90eb19c6d07b
train=0.1149: 100%|██████████| 3000/3000 [01:14<00:00, 40.52it/s]
[I 2025-02-19 16:29:06,189] Trial 0 finished with value: 0.19177547097206116 and parameters: {'d_hidden': 128, 'n_hidden': 5, 'dropout_rate': 0.05746146364703811, 'activation': 'leaky_relu', 'learning_rate': 0.0010619398342566597, 'clip_norm': True, 'cosine_sched': True}. Best is trial 0 with value: 0.19177547097206116.
train=0.2347: 100%|██████████| 3000/3000 [00:18<00:00, 160.85it/s]
[I 2025-02-19 16:29:26,640] Trial 1 finished with value: 0.19550496339797974 and parameters: {'d_hidden': 64, 'n_hidden': 1, 'dropout_rate': 0.0026558335486737746, 'activation': 'relu', 'learning_rate': 0.0057678914821090235, 'clip_norm': True, 'cosine_sched': True}. Best is trial 0 with value: 0.19177547097206116.
train=0.2620: 100%|██████████| 3000/3000 [01:01<00:00, 48.79it/s]
[I 2025-02-19 16:30:28,922] Trial 2 finished wit

# MLP + CNN

In [13]:
X_particle, X_field, Y_particle, Y_field = data.get_offline_regression_data(train_dict, y_labels=["P"])
X_particle_vali, X_field_vali, Y_particle_vali, Y_field_vali = data.get_offline_regression_data(test_dict, y_labels=["P"])

def mlp_cnn_training(
    # net
    mlp_d_hidden,
    mlp_n_hidden,
    mlp_dropout_rate,
    mlp_activation,
    d_latent,
    cnn_d_hidden,
    cnn_n_hidden,
    # optimizer
    n_steps,
    learning_rate,
    clip_norm,
    cosine_sched,
):
    if mlp_activation == "relu":
        activation = jax.nn.relu
    elif mlp_activation == "leaky_relu":
        activation = jax.nn.leaky_relu
    elif mlp_activation == "sigmoid":
        activation = jax.nn.sigmoid
    
    mlp = MLP(
        d_in=X_particle.shape[-1], 
        d_out=d_latent, 
        d_hidden=mlp_d_hidden, 
        n_hidden=mlp_n_hidden, 
        activation=activation,
        rngs=nnx.Rngs(0)
    )
    
    cnn = CNN(
        d_in=X_field.shape[-1], 
        d_out=d_latent,
        d_hidden=cnn_d_hidden,
        n_hidden=cnn_n_hidden,
        kernel_size=(3, 3, 3),
        strides=1,
        rngs=nnx.Rngs(0)
    )
    
    model = HybridNet(
        mlp,
        cnn,
        d_out=Y_particle.shape[-1],
        rngs=nnx.Rngs(0),
        batch_axis=False,
    )    

    vali_loss = train_loop(
        model, 
        (train_dict["gas_poss"], X_particle, X_field), 
        Y_particle,
        (test_dict["gas_poss"], X_particle_vali, X_field_vali), 
        Y_particle_vali,
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )

    return vali_loss

def mlp_cnn_objective(trial):
    mlp_d_hidden = trial.suggest_categorical("mlp_d_hidden", [16, 32, 64, 128])
    mlp_n_hidden = trial.suggest_int("mlp_n_hidden", 1, 8)
    mlp_dropout_rate = trial.suggest_float("mlp_dropout_rate", 1e-3, 1e-1, log=True)
    mlp_activation = trial.suggest_categorical("mlp_activation", ["relu", "sigmoid", "leaky_relu"])
    d_latent = trial.suggest_categorical("d_latent", [4, 8, 16])
    cnn_d_hidden = trial.suggest_categorical("cnn_d_hidden", [8, 16, 32])
    cnn_n_hidden = trial.suggest_categorical("cnn_n_hidden", [1, 2, 3])
                               
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    clip_norm = trial.suggest_categorical("clip_norm", [True, False])
    cosine_sched = trial.suggest_categorical("cosine_sched", [True, False])

    n_steps = 3_000
    # n_steps = trial.suggest_int("train_steps", 1000, 100000, log=True)

    val_loss = mlp_cnn_training(
        # net
        mlp_d_hidden,
        mlp_n_hidden,
        mlp_dropout_rate,
        mlp_activation,
        d_latent,
        cnn_d_hidden,
        cnn_n_hidden,
        # optimizer
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )
    return val_loss

In [14]:
study = optuna.create_study(direction="minimize")
study.optimize(mlp_cnn_objective, n_trials=20)

[I 2025-02-19 16:45:49,840] A new study created in memory with name: no-name-ee3f6b54-b794-4c49-80fe-46b8e438aed0
train=0.1605: 100%|██████████| 3000/3000 [01:51<00:00, 26.94it/s]
[I 2025-02-19 16:47:44,567] Trial 0 finished with value: 0.1846800446510315 and parameters: {'mlp_d_hidden': 32, 'mlp_n_hidden': 5, 'mlp_dropout_rate': 0.0034692136421046603, 'mlp_activation': 'leaky_relu', 'd_latent': 16, 'cnn_d_hidden': 16, 'cnn_n_hidden': 2, 'learning_rate': 0.004483420878394885, 'clip_norm': True, 'cosine_sched': False}. Best is trial 0 with value: 0.1846800446510315.
train=0.2546: 100%|██████████| 3000/3000 [01:38<00:00, 30.39it/s]
[I 2025-02-19 16:49:26,577] Trial 1 finished with value: 0.21186695992946625 and parameters: {'mlp_d_hidden': 128, 'mlp_n_hidden': 2, 'mlp_dropout_rate': 0.0026707194313075736, 'mlp_activation': 'relu', 'd_latent': 8, 'cnn_d_hidden': 32, 'cnn_n_hidden': 3, 'learning_rate': 0.00018049484701078482, 'clip_norm': False, 'cosine_sched': True}. Best is trial 0 with 

# GNN

In [15]:
X, Y = graph.get_graph(
    train_dict,
    y_labels=["P"],
    k=16, 
    # boxsize=mesh_per_dim
)

X_vali, Y_vali = graph.get_graph(
    test_dict,
    y_labels=["P"],
    k=16, 
    # boxsize=mesh_per_dim
)

def gnn_training(
    # net
    d_query,
    n_hidden,
    activation,
    query_activation,
    logit_activation,
    final_projection,
    # optimizer
    n_steps,
    learning_rate,
    clip_norm,
    cosine_sched,
):
    if activation == "relu":
        activation = jax.nn.relu
    elif activation == "leaky_relu":
        activation = jax.nn.leaky_relu
    elif activation == "sigmoid":
        activation = jax.nn.sigmoid

    model = AttentionGNN(
        d_node=X[0].nodes.shape[-1],
        d_edge=X[0].edges.shape[-1],
        d_query=d_query,
        n_hidden=n_hidden,
        d_out=Y.shape[-1],
        rngs=nnx.Rngs(0),
    )

    vali_loss = train_loop(
        model, 
        (X,), 
        Y,
        (X_vali,),
        Y_vali,
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )

    return vali_loss


def gnn_objective(trial):
    d_query = trial.suggest_categorical("d_query", [8, 16])
    n_hidden = trial.suggest_int("n_hidden", 1, 6)
    activation = trial.suggest_categorical("activation", ["relu", "sigmoid", "leaky_relu"])
    query_activation = trial.suggest_categorical("query_activation", [True, False])
    logit_activation = trial.suggest_categorical("logit_activation", [True, False])
    final_projection = trial.suggest_categorical("final_projection", [True, False])

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    clip_norm = trial.suggest_categorical("clip_norm", [True, False])
    cosine_sched = trial.suggest_categorical("cosine_sched", [True, False])

    n_steps = 3_000
    # n_steps = trial.suggest_int("train_steps", 1000, 100000, log=True)

    val_loss = gnn_training(
        d_query,
        n_hidden,
        activation,
        query_activation,
        logit_activation,
        final_projection,
        n_steps,
        learning_rate,
        clip_norm,
        cosine_sched,
    )
    return val_loss

In [16]:
study = optuna.create_study(direction="minimize")
study.optimize(gnn_objective, n_trials=20)

[I 2025-02-19 17:13:01,316] A new study created in memory with name: no-name-b28cb530-61d7-4930-887b-a944d95d7945
train=0.1664: 100%|██████████| 3000/3000 [03:31<00:00, 14.18it/s]
[I 2025-02-19 17:16:36,210] Trial 0 finished with value: 0.1557750403881073 and parameters: {'d_query': 16, 'n_hidden': 2, 'activation': 'sigmoid', 'query_activation': True, 'logit_activation': True, 'final_projection': False, 'learning_rate': 0.0013483256858140755, 'clip_norm': False, 'cosine_sched': False}. Best is trial 0 with value: 0.1557750403881073.
train=0.1693: 100%|██████████| 3000/3000 [01:38<00:00, 30.51it/s]
[I 2025-02-19 17:18:17,570] Trial 1 finished with value: 0.16122479736804962 and parameters: {'d_query': 8, 'n_hidden': 1, 'activation': 'sigmoid', 'query_activation': True, 'logit_activation': True, 'final_projection': True, 'learning_rate': 0.007911207822866606, 'clip_norm': False, 'cosine_sched': True}. Best is trial 0 with value: 0.1557750403881073.
train=0.2683: 100%|██████████| 3000/300